# Data Mining

In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import os, json

from matminer.featurizers.composition import ElementFraction, ElementProperty
from matminer.featurizers.conversions import StrToComposition


In [2]:
# Lightweight “MXene-like” composition filter (no structure needed).
# Goal: transition metal + (C or N), avoid A-layer elements (MAX phase A: e.g., Al, Si, Ga, Ge, In, Sn, Pb, etc.)
TRANSITION_METALS = {
    "Sc","Ti","V","Cr","Mn","Fe","Co","Ni","Cu","Zn",
    "Y","Zr","Nb","Mo","Tc","Ru","Rh","Pd","Ag","Cd",
    "Hf","Ta","W","Re","Os","Ir","Pt","Au","Hg"
}
A_LAYER = {"Al","Si","P","Ga","Ge","As","In","Sn","Sb","Tl","Pb","Bi"}

TERMINATION_CANDIDATES = {"O","F","Cl","Br","I","H","S"}  # common surface terminations

def is_mxene_like(comp):
    """
    Heuristic:
      - has >=1 transition metal
      - has C or N
      - ideally 2D in MP (handled in Path A), but here we filter by composition only
      - avoids A-layer elements typical of MAX phases (post-etching MXenes should lack them)
    """
    elts = set([el.symbol for el in comp.elements])
    has_tm = any(e in TRANSITION_METALS for e in elts)
    has_x = ("C" in elts) or ("N" in elts)
    has_a_layer = any(e in A_LAYER for e in elts)
    return has_tm and has_x and not has_a_layer


In [3]:
# Pull data from Materials Project & cache to CSV
os.environ["MP_API_KEY"] = "9LfTPxDWpGGUQYOdvrYQt52uKViOI09r"
os.makedirs("./data", exist_ok=True)
DATA_CSV = "./data/mxene_mp.csv"

def fetch_from_mp_api(max_docs=10000):
    from mp_api.client import MPRester
    from pymatgen.core.composition import Composition

    key = os.getenv("MP_API_KEY", None)
    if key is None or not key.strip():
        raise RuntimeError("MP_API_KEY not set. Set it in your shell or inside the notebook via os.environ.")

    fields = [
        "material_id", "formula_pretty", "composition",
        "formation_energy_per_atom", "band_gap",
        "nelements", "elements"
    ]

    # Two broad queries (C-containing and N-containing) and filter to MXene-like in Python.
    with MPRester(api_key=key) as mpr:
        docs_C = mpr.summary.search(
            elements=["C"],                      # contains carbon
            exclude_elements=list(A_LAYER),      # avoid A-layer elements
            fields=fields,
            chunk_size=1000,                      # pagination controls
            num_chunks=10
        )
        docs_N = mpr.summary.search(
            elements=["N"],                      # contains nitrogen
            exclude_elements=list(A_LAYER),
            fields=fields,
            chunk_size=1000,
            num_chunks=10
        )

    # Merge & deduplicate by material_id
    merged = {}
    for d in list(docs_C) + list(docs_N):
        merged[str(d.material_id)] = d
    docs = list(merged.values())

    rows = []
    for d in docs[:max_docs]:
        comp_dict = d.composition
        if not comp_dict:
            continue
        comp = Composition(comp_dict)

        # Heuristic MXene-like filter (composition-only)
        if not is_mxene_like(comp):
            continue

        fe = d.formation_energy_per_atom
        bg = d.band_gap
        if fe is None and bg is None:
            continue

        rows.append({
            "material_id": str(d.material_id),
            "formula": d.formula_pretty,
            "composition": json.dumps(comp.as_dict()),
            "formation_energy_per_atom": fe,
            "band_gap": bg
        })

    df = (pd.DataFrame(rows)
        .drop_duplicates(subset=["material_id"])
        .dropna(subset=["formula", "formation_energy_per_atom", "band_gap"], how="all")
        )
    print(f"Fetched {len(df)} MXene-like rows from MP (after filtering).")
    return df

df_raw = None
try:
    df_raw = fetch_from_mp_api(max_docs=10000)
    df_raw.to_csv(DATA_CSV, index=False)
    print(f"Cached to {DATA_CSV}")
except Exception as e:
    print("Online fetch skipped or failed:", e)


/tmp/ipykernel_917171/7133499.py:22: DeprecationWarning: Accessing summary data through MPRester.summary is deprecated. Please use MPRester.materials.summary instead.
  docs_C = mpr.summary.search(


Retrieving SummaryDoc documents:   0%|          | 0/6293 [00:00<?, ?it/s]

/tmp/ipykernel_917171/7133499.py:29: DeprecationWarning: Accessing summary data through MPRester.summary is deprecated. Please use MPRester.materials.summary instead.
  docs_N = mpr.summary.search(


Retrieving SummaryDoc documents:   0%|          | 0/8443 [00:00<?, ?it/s]

Fetched 6419 MXene-like rows from MP (after filtering).
Cached to ./data/mxene_mp.csv


# Feature Extraction

In [4]:
df = pd.read_csv(DATA_CSV)
df.head()

,material_id,formula,composition,formation_energy_per_atom,band_gap
0,mp-1178822,VC3,"{""V"": 8.0, ""C"": 24.0}",0.595332,0.0000
1,mp-1078579,Fe4C,"{""Fe"": 8.0, ""C"": 2.0}",0.133002,0.0000
2,mp-1201082,Y15C19,"{""Y"": 30.0, ""C"": 38.0}",-0.287071,0.0000
3,mp-1189780,Os2C3,"{""Os"": 8.0, ""C"": 12.0}",1.224335,0.2375
4,mp-1197234,Ir4C5,"{""Ir"": 16.0, ""C"": 20.0}",1.032691,0.5691


In [5]:
# (1) Convert string formula → pymatgen Composition object
stc = StrToComposition(target_col_id="composition_obj")
df_feat = stc.featurize_dataframe(df, "formula", ignore_errors=True)

# (2) Compute elemental fractions
ef = ElementFraction()
df_feat = ef.featurize_dataframe(df_feat, "composition_obj", ignore_errors=True)

# (3) Magpie elemental statistics
magpie = ElementProperty.from_preset(preset_name="magpie")
df_feat = magpie.featurize_dataframe(df_feat, "composition_obj", ignore_errors=True)

# Collect feature columns
non_feature_cols = {
    "material_id","formula","composition","composition_obj",
    "formation_energy_per_atom","band_gap"
}
feature_cols = [c for c in df_feat.columns if c not in non_feature_cols]
print("n_features:", len(feature_cols))


StrToComposition:   0%|          | 0/6419 [00:00<?, ?it/s]

ElementFraction:   0%|          | 0/6419 [00:00<?, ?it/s]

/home/adroit/miniconda3/envs/material/lib/python3.10/site-packages/matminer/utils/data.py:326: UserWarning: MagpieData(impute_nan=False):
In a future release, impute_nan will be set to True by default.
                    This means that features that are missing or are NaNs for elements
                    from the data source will be replaced by the average of that value
                    over the available elements.
                    This avoids NaNs after featurization that are often replaced by
                    dataset-dependent averages.
  warnings.warn(f"{self.__class__.__name__}(impute_nan=False):\n" + IMPUTE_NAN_WARNING)


ElementProperty:   0%|          | 0/6419 [00:00<?, ?it/s]

n_features: 250


In [6]:
# See and save
df_feat.to_csv("./data/train_mp.csv", index=False)
print("Saved featurized data to train_mp.csv")
df_feat.head()


Saved featurized data to train_mp.csv


,material_id,formula,composition,formation_energy_per_atom,band_gap,composition_obj,H,He,Li,Be,...,MagpieData range GSmagmom,MagpieData mean GSmagmom,MagpieData avg_dev GSmagmom,MagpieData mode GSmagmom,MagpieData minimum SpaceGroupNumber,MagpieData maximum SpaceGroupNumber,MagpieData range SpaceGroupNumber,MagpieData mean SpaceGroupNumber,MagpieData avg_dev SpaceGroupNumber,MagpieData mode SpaceGroupNumber
0,mp-1178822,VC3,"{""V"": 8.0, ""C"": 24.0}",0.595332,0.0000,"(V, C)",0.0,0,0.0,0,...,0.000000,0.00000,0.000000,0.000000,194.0,229.0,35.0,202.750000,13.125000,194.0
1,mp-1078579,Fe4C,"{""Fe"": 8.0, ""C"": 2.0}",0.133002,0.0000,"(Fe, C)",0.0,0,0.0,0,...,2.110663,1.68853,0.675412,2.110663,194.0,229.0,35.0,222.000000,11.200000,229.0
2,mp-1201082,Y15C19,"{""Y"": 30.0, ""C"": 38.0}",-0.287071,0.0000,"(Y, C)",0.0,0,0.0,0,...,0.000000,0.00000,0.000000,0.000000,194.0,194.0,0.0,194.000000,0.000000,194.0
3,mp-1189780,Os2C3,"{""Os"": 8.0, ""C"": 12.0}",1.224335,0.2375,"(Os, C)",0.0,0,0.0,0,...,0.000000,0.00000,0.000000,0.000000,194.0,194.0,0.0,194.000000,0.000000,194.0
4,mp-1197234,Ir4C5,"{""Ir"": 16.0, ""C"": 20.0}",1.032691,0.5691,"(Ir, C)",0.0,0,0.0,0,...,0.000000,0.00000,0.000000,0.000000,194.0,225.0,31.0,207.777778,15.308642,194.0
